In [ ]:
import sys
print(sys.executable)

In [ ]:
from AutoREACTER.initialization import Initialization
from AutoREACTER.input_parser import InputParser
from AutoREACTER.cache import GetCacheDir, RunDirectoryManager, RetentionCleanup
from AutoREACTER.detectors.functional_groups_detector import FunctionalGroupsDetector
from AutoREACTER.detectors.reaction_detector import ReactionDetector
from AutoREACTER.reaction_template_builder.run_reaction_template_pipeline import ReactionTemplatePipeline
from rdkit import Chem
from rdkit.Chem import Draw
import json

Initialization()
input_parser = InputParser()

cache_dir = GetCacheDir().staging_dir
dated_cache_dir = RunDirectoryManager.make_dated_run_dir(cache_dir, chdir_to="none")
# #future use
# RunDirectoryManager.copy_into_run(cache_dir, dated_cache_dir)




In [ ]:
# if you want to clean up the cache, uncomment run this cell
# This will delete all cached data based on the date

# RetentionCleanup.run(GetCacheDir().cache_base_dir)

In [ ]:
with open("example_1_inputs_count_mode.json", "r") as f:
    input_data = json.load(f)

validated_inputs = input_parser.validate_inputs(input_data)
initial_molecules, legends = input_parser.molecule_representation_of_initial_molecules(validated_inputs)
initial_molecules = Draw.MolsToGridImage(initial_molecules, molsPerRow=3, subImgSize=(400, 400), legends=legends)
initial_molecules

In [ ]:
# or user can use a json file to read inputs

# inputs_parser = InputParser()          # validates once already
# validated_inputs = inputs_parser.to_dict()       # validates again, returns dict

default_cache_dir = GetCacheDir().cache_base_dir
print(f"Default cache directory: {default_cache_dir}")



In [ ]:
# Example of using retention_cleanup
# This function will delete files in the cache directory that are older than the 
# specified number of days (e.g., 30 days)
# retention_cleanup(default_cache_dir, retention_days=30)

In [ ]:
# Detector Has to called Now after the inputs are validated, because it needs the input dict to run the detection workflow
functional_groups_detector = FunctionalGroupsDetector()
functional_groups, functional_groups_imgs = 
    functional_groups_detector.functional_groups_detector(
        validated_inputs.monomers
    )

for fg_img in functional_groups_imgs:
    print(fg_img.indexes_to_highlight)

In [ ]:
img = functional_groups_detector.molecules_to_visualization(functional_groups_imgs)
img


In [ ]:
reaction_detector = ReactionDetector()
reaction_instances = reaction_detector.reaction_detector(functional_groups)
img = reaction_detector.create_reaction_image_grid(reaction_instances)
img

In [ ]:
selected_reactions = reaction_detector.reaction_selection(reaction_instances)

In [ ]:
%%capture captured_output
# execute_pipeline may live in the same module as ReactionTemplatePipeline; import if needed.
from AutoREACTER.reaction_template_builder.run_reaction_template_pipeline import execute_pipeline

execute_pipeline(
    detected_reactions=detected_reactions,
    retain_smiles=non_monomers,
    cache=cache_dir,
)


In [ ]:
from pathlib import Path

with open(Path(cache_dir) / "output.txt", "w") as f:
    f.write(captured_output.stdout)

print(f"Output written to {Path(cache_dir) / 'output.txt'}")
